# Notebook 01 — Data Acquisition

**Goal:** Establish data access patterns, explore the schema, and confirm coverage across all MBTA lines.

Data source: [MBTA LAMP Public Data](https://performancedata.mbta.com)  
All data is read directly from public URLs — no download required upfront.

## Sections
1. Setup & Imports
2. Index File — What dates are available?
3. Helper Functions — Load single day / date range
4. Schema Exploration — Columns, dtypes, nulls
5. Coverage Check — Lines, routes, Green Line B
6. Save Sample for Downstream Notebooks

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from pathlib import Path
from datetime import date, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Project paths
ROOT        = Path('..').resolve()
RAW_DIR     = ROOT / 'data' / 'raw' / 'subway_perf'
GTFS_DIR    = ROOT / 'data' / 'raw' / 'gtfs_static'
PROC_DIR    = ROOT / 'data' / 'processed'

# MBTA LAMP base URL
BASE_URL    = 'https://performancedata.mbta.com/lamp'
PERF_URL    = f'{BASE_URL}/subway-on-time-performance-v1'
INDEX_URL   = f'{PERF_URL}/index.csv'

print('Paths OK:', ROOT.exists())

## 2. Index File — What Dates Are Available?

In [ ]:
# Load the master index — lists every available date and its file URL
index = pd.read_csv(INDEX_URL, parse_dates=['service_date'])
index = index.sort_values('service_date').reset_index(drop=True)

print(f'Total available dates : {len(index):,}')
print(f'Earliest date         : {index.service_date.min().date()}')
print(f'Latest date           : {index.service_date.max().date()}')
print(f'Total size (GB)       : {index.size_bytes.sum() / 1e9:.2f}')
index.head()

In [ ]:
# Filter to our analysis window: 2022-01-01 ~ 2026-05-31 (exclude COVID era)
mask = (index.service_date >= '2022-01-01') & (index.service_date <= '2026-05-31')
index_22_26 = index[mask].reset_index(drop=True)

print(f'Dates in 2022-2026/05 : {len(index_22_26):,}')
print(f'Size in window (GB)   : {index_22_26.size_bytes.sum() / 1e9:.2f}')

# Check for gaps
all_dates = pd.date_range('2022-01-01', '2026-05-31', freq='D')
missing   = set(all_dates.date) - set(index_22_26.service_date.dt.date)
print(f'Missing dates         : {len(missing)}')
if missing:
    print('  Sample missing:', sorted(missing)[:10])


In [ ]:
# Visualise file size over time (proxy for service volume)
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(index_22_26.service_date, index_22_26.size_bytes / 1e6, lw=0.7, color='steelblue')
ax.set_xlabel('Date')
ax.set_ylabel('File size (MB)')
ax.set_title('Daily Parquet file size — proxy for service data volume (2022–2026/05)')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f MB'))
plt.tight_layout()
plt.show()


## 3. Helper Functions — Load Single Day / Date Range

In [ ]:
def load_day(service_date: str, columns: list = None) -> pd.DataFrame:
    """
    Load one day of subway performance data directly from MBTA LAMP.
    
    Parameters
    ----------
    service_date : str  e.g. '2024-01-15'
    columns      : list of column names to load (None = all)
    
    Returns
    -------
    pd.DataFrame
    """
    url = f'{PERF_URL}/{service_date}-subway-on-time-performance-v1.parquet'
    df  = pd.read_parquet(url, columns=columns)
    df['service_date'] = pd.to_datetime(service_date)
    return df


def load_date_range(
    start: str,
    end: str,
    columns: list = None,
    sample_frac: float = 1.0,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Load multiple days of data, concatenating into one DataFrame.
    Uses index.csv so only dates that actually exist are fetched.
    
    Parameters
    ----------
    start        : str  'YYYY-MM-DD'
    end          : str  'YYYY-MM-DD'
    columns      : list of columns to load (None = all)
    sample_frac  : float  0-1, fraction of rows to keep per day (for quick tests)
    verbose      : bool   print progress
    """
    mask    = (index.service_date >= start) & (index.service_date <= end)
    dates   = index.loc[mask, 'service_date'].dt.strftime('%Y-%m-%d').tolist()
    frames  = []

    for i, d in enumerate(dates):
        if verbose and i % 30 == 0:
            print(f'  Loading {d}  ({i+1}/{len(dates)})')
        try:
            df = load_day(d, columns=columns)
            if sample_frac < 1.0:
                df = df.sample(frac=sample_frac, random_state=42)
            frames.append(df)
        except Exception as e:
            if verbose:
                print(f'  WARNING: {d} failed — {e}')

    result = pd.concat(frames, ignore_index=True)
    if verbose:
        print(f'Done. Loaded {len(result):,} rows from {len(frames)} days.')
    return result


print('Helper functions defined.')

## 4. Schema Exploration — Columns, dtypes, Nulls
Load a single day first to inspect the schema cheaply.

In [ ]:
# Load one representative weekday
sample = load_day('2024-03-12')  # Tuesday, typical weekday
print(f'Rows: {len(sample):,}   Columns: {sample.shape[1]}')
sample.head(3)

In [ ]:
# Column overview: dtype + null rate
schema = pd.DataFrame({
    'dtype'    : sample.dtypes.astype(str),
    'null_pct' : (sample.isna().mean() * 100).round(2),
    'n_unique'  : sample.nunique(),
    'sample_val': sample.iloc[0]
})
schema.sort_values('null_pct', ascending=False)

In [ ]:
# Key columns we'll use — confirm they exist and look sensible
KEY_COLS = [
    'service_date', 'route_id', 'direction_id',
    'stop_id', 'parent_station',
    'vehicle_id', 'trip_id',
    'move_timestamp', 'stop_timestamp',
    'travel_time_seconds', 'dwell_time_seconds',
    'headway_branch_seconds', 'headway_trunk_seconds',
    'scheduled_arrival_time', 'scheduled_departure_time',
]

missing_cols = [c for c in KEY_COLS if c not in sample.columns]
print('Missing key columns:', missing_cols if missing_cols else 'None — all present')

sample[KEY_COLS].describe(include='all').T

## 5. Coverage Check — Lines, Routes, Green Line B

In [ ]:
# What route_ids are in the data?
route_counts = sample.groupby('route_id').size().sort_values(ascending=False)
print('All route_ids in this day:')
print(route_counts.to_string())

In [ ]:
# Map route_id to line name for readability
ROUTE_LINE_MAP = {
    'Red'    : 'Red Line',
    'Orange' : 'Orange Line',
    'Blue'   : 'Blue Line',
    'Green-B': 'Green Line B',
    'Green-C': 'Green Line C',
    'Green-D': 'Green Line D',
    'Green-E': 'Green Line E',
    'Mattapan': 'Mattapan Trolley',
}

sample['line'] = sample['route_id'].map(ROUTE_LINE_MAP).fillna(sample['route_id'])

line_summary = (
    sample.groupby('line')
    .agg(
        n_trips        = ('trip_id', 'nunique'),
        n_stops        = ('stop_id', 'nunique'),
        avg_travel_sec = ('travel_time_seconds', 'mean'),
        avg_dwell_sec  = ('dwell_time_seconds', 'mean'),
        avg_headway_sec= ('headway_branch_seconds', 'mean'),
    )
    .round(1)
)
line_summary

In [ ]:
# Green Line B — check surface vs. underground stops
green_b = sample[sample['route_id'] == 'Green-B'].copy()
print(f'Green Line B rows in this day: {len(green_b):,}')
print(f'Unique stops: {green_b.stop_id.nunique()}')
print()

# Surface stops on B Branch (west of Kenmore, above ground)
# These stop_ids correspond to street-level stations
SURFACE_STOP_PREFIXES = [
    'place-bland', 'place-brico', 'place-harvd', 'place-patk',
    'place-babck', 'place-plsgr', 'place-sthst', 'place-chswk',
    'place-sumav', 'place-grigg', 'place-alsgr', 'place-wrnst',
    'place-wascm', 'place-bc',
]

green_b['is_surface'] = green_b['parent_station'].isin(SURFACE_STOP_PREFIXES)

surface_compare = (
    green_b.groupby('is_surface')
    .agg(
        avg_dwell_sec   = ('dwell_time_seconds', 'mean'),
        median_dwell_sec= ('dwell_time_seconds', 'median'),
        avg_headway_sec = ('headway_branch_seconds', 'mean'),
        n_events        = ('trip_id', 'count'),
    )
    .round(2)
)
surface_compare.index = surface_compare.index.map({True: 'Surface (B line)', False: 'Underground/Tunnel'})
print('Dwell time: Surface vs Underground')
surface_compare

In [ ]:
# Distribution of key metrics — quick visual sanity check
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ('travel_time_seconds', 'Travel Time (sec)', 'steelblue'),
    ('dwell_time_seconds',  'Dwell Time (sec)',  'darkorange'),
    ('headway_branch_seconds', 'Branch Headway (sec)', 'seagreen'),
]

for ax, (col, label, color) in zip(axes, metrics):
    data = sample[col].dropna()
    # Clip extreme outliers for visualisation
    p99 = data.quantile(0.99)
    ax.hist(data[data <= p99], bins=60, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(label)
    ax.set_xlabel('Seconds')
    ax.set_ylabel('Count')
    ax.axvline(data.median(), color='black', ls='--', lw=1.2, label=f'Median: {data.median():.0f}s')
    ax.legend(fontsize=9)

fig.suptitle('Distribution of Key Metrics — 2024-03-12 (one weekday)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Load One Month Sample & Save for Downstream Notebooks
We'll cache a representative 1-month sample so other notebooks can load instantly without re-fetching.

In [ ]:
# Define the columns we actually need downstream (reduces memory significantly)
KEEP_COLS = [
    'service_date',
    'route_id',
    'direction_id',
    'trip_id',
    'vehicle_id',
    'stop_id',
    'parent_station',
    'stop_sequence',
    'move_timestamp',
    'stop_timestamp',
    'travel_time_seconds',
    'dwell_time_seconds',
    'headway_branch_seconds',
    'headway_trunk_seconds',
    'scheduled_arrival_time',
    'scheduled_departure_time',
    'scheduled_travel_time_seconds',
]

# Only keep columns that actually exist in the data
KEEP_COLS = [c for c in KEEP_COLS if c in sample.columns]
print(f'Columns to keep: {len(KEEP_COLS)}')
print(KEEP_COLS)

In [ ]:
# Load Jan 2024 as a representative sample (31 days)
# This will take ~1-2 minutes
print('Loading January 2024...')
jan2024 = load_date_range(
    start='2024-01-01',
    end='2024-01-31',
    columns=KEEP_COLS,
    verbose=True
)

# Add line label
jan2024['line'] = jan2024['route_id'].map(ROUTE_LINE_MAP).fillna(jan2024['route_id'])

print(f'\nSample shape: {jan2024.shape}')
print(f'Memory usage: {jan2024.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Save to processed/ for fast loading in downstream notebooks
out_path = PROC_DIR / 'sample_jan2024.parquet'
jan2024.to_parquet(out_path, index=False)
print(f'Saved: {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)')

In [ ]:
# Quick summary by line for the saved sample
(
    jan2024.groupby('line')
    .agg(
        n_days         = ('service_date', 'nunique'),
        n_trips        = ('trip_id',      'nunique'),
        total_events   = ('trip_id',      'count'),
        avg_dwell_sec  = ('dwell_time_seconds',     'mean'),
        avg_headway_sec= ('headway_branch_seconds', 'mean'),
    )
    .round(1)
    .sort_values('total_events', ascending=False)
)

## 7. Build Full Aggregate Dataset (2022–2026/05)

Process all 1,612 days one at a time, compute `station × hour × route` statistics per day, save monthly checkpoints to disk.

- **Only 6 columns loaded per day** → low memory footprint
- **Checkpoint per month** → safe to re-run if interrupted
- **Output:** `data/processed/aggregate_full.parquet` (~4–5 M rows, ~100 MB)

> This takes ~20–40 min on first run. Re-running skips already-processed months.

In [ ]:
AGG_COLS = [
    'route_id', 'parent_station', 'stop_timestamp',
    'headway_branch_seconds', 'dwell_time_seconds', 'travel_time_seconds',
]

ROUTE_LINE_MAP_LOCAL = {
    'Red': 'Red Line', 'Orange': 'Orange Line', 'Blue': 'Blue Line',
    'Green-B': 'Green Line B', 'Green-C': 'Green Line C',
    'Green-D': 'Green Line D', 'Green-E': 'Green Line E',
    'Mattapan': 'Mattapan Trolley',
}

def compute_daily_agg(df, service_date):
    """Aggregate one day of stop events to station × hour × route level."""
    df = df.copy()
    # Extract hour from Unix timestamp (Eastern time)
    df['hour'] = (
        pd.to_datetime(df['stop_timestamp'], unit='s', utc=True)
        .dt.tz_convert('America/New_York')
        .dt.hour
    )
    df['is_bunched'] = df['headway_branch_seconds'] < 120
    df['line'] = df['route_id'].map(ROUTE_LINE_MAP_LOCAL).fillna(df['route_id'])

    agg = (
        df.groupby(['route_id', 'line', 'parent_station', 'hour'])
        .agg(
            n_events        = ('stop_timestamp',          'count'),
            n_headway       = ('headway_branch_seconds',  'count'),
            bunching_events = ('is_bunched',              'sum'),
            median_headway  = ('headway_branch_seconds',  'median'),
            median_dwell    = ('dwell_time_seconds',      'median'),
            mean_dwell      = ('dwell_time_seconds',      'mean'),
            p75_dwell       = ('dwell_time_seconds',      lambda x: x.quantile(0.75)),
            median_travel   = ('travel_time_seconds',     'median'),
        )
        .reset_index()
    )

    sdate              = pd.to_datetime(service_date)
    agg['service_date']= sdate
    agg['year']        = sdate.year
    agg['month']       = sdate.month
    agg['dow']         = sdate.dayofweek          # 0=Mon
    agg['is_weekend']  = sdate.dayofweek >= 5
    agg['bunching_rate'] = (
        agg['bunching_events'] / agg['n_headway'].replace(0, pd.NA)
    )
    return agg

print('compute_daily_agg() defined.')


In [ ]:
import os

CKPT_DIR = PROC_DIR / 'agg_checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# All dates in analysis window (already filtered in cell above)
dates = index_22_26['service_date'].dt.strftime('%Y-%m-%d').tolist()

# Group by year-month for checkpointing
from itertools import groupby

def ym_key(d):
    return d[:7]   # 'YYYY-MM'

processed, skipped, failed = 0, 0, 0

for ym, group in groupby(dates, key=ym_key):
    ckpt_path = CKPT_DIR / f'{ym}.parquet'
    day_list  = list(group)

    if ckpt_path.exists():
        skipped += len(day_list)
        continue

    month_frames = []
    for d in day_list:
        url = f'{PERF_URL}/{d}-subway-on-time-performance-v1.parquet'
        try:
            raw = pd.read_parquet(url, columns=AGG_COLS)
            month_frames.append(compute_daily_agg(raw, d))
            processed += 1
            del raw
        except Exception as e:
            print(f'  WARNING: {d} — {e}')
            failed += 1

    if month_frames:
        pd.concat(month_frames, ignore_index=True).to_parquet(ckpt_path, index=False)
        print(f'  Saved checkpoint: {ym}  ({len(month_frames)} days)')

print(f'\nDone.  Processed={processed}  Skipped(cached)={skipped}  Failed={failed}')


In [ ]:
# Combine all monthly checkpoints into one file
ckpt_files = sorted(CKPT_DIR.glob('*.parquet'))
print(f'Combining {len(ckpt_files)} monthly checkpoints...')

agg_full = pd.concat(
    [pd.read_parquet(f) for f in ckpt_files],
    ignore_index=True
)

out_path = PROC_DIR / 'aggregate_full.parquet'
agg_full.to_parquet(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Shape: {agg_full.shape}')
print(f'Size:  {out_path.stat().st_size / 1e6:.1f} MB')
print(f'Date range: {agg_full.service_date.min().date()} → {agg_full.service_date.max().date()}')
print(f'Years: {sorted(agg_full.year.unique())}')
print()
print(agg_full.groupby('year').agg(
    days  = ('service_date', 'nunique'),
    rows  = ('n_events',     'count'),
    total_events = ('n_events', 'sum'),
).to_string())


## 8. Build Strategic Row-Level Sample (2024–2026/05, Complete)

Load **every month** of 2024, 2025, and 2026 Jan–May — one month at a time.
Each month is saved as its own parquet file in `data/processed/strategic/`.

- Peak memory: only **1 month at a time** (~500 MB)
- Total on disk: ~250 MB (Parquet compressed)
- Covers all holidays, seasonal effects, and complete years
- Downstream notebooks load individual months or filter by date as needed

> Takes ~15–20 min. Already-saved months are skipped on re-run.

In [ ]:
import pandas as pd
from pathlib import Path

STRATEGIC_DIR = PROC_DIR / 'strategic'
STRATEGIC_DIR.mkdir(exist_ok=True)

# All months from 2024-01 to 2026-05
months = pd.period_range('2024-01', '2026-05', freq='M')

processed, skipped, failed_days = 0, 0, 0

for period in months:
    year, month = period.year, period.month
    ckpt = STRATEGIC_DIR / f'{year}-{month:02d}.parquet'

    if ckpt.exists():
        print(f'  Skip {year}-{month:02d} (already saved)')
        skipped += 1
        continue

    start = f'{year}-{month:02d}-01'
    end   = period.to_timestamp('M').strftime('%Y-%m-%d')

    mask  = (index.service_date >= start) & (index.service_date <= end)
    dates_m = index.loc[mask, 'service_date'].dt.strftime('%Y-%m-%d').tolist()

    frames = []
    for d in dates_m:
        url = f'{PERF_URL}/{d}-subway-on-time-performance-v1.parquet'
        try:
            tmp = pd.read_parquet(url, columns=[c for c in KEEP_COLS if c != 'service_date'])
            tmp['service_date'] = pd.to_datetime(d)
            frames.append(tmp)
        except Exception as e:
            print(f'    WARNING: {d} — {e}')
            failed_days += 1

    if frames:
        month_df = pd.concat(frames, ignore_index=True)
        month_df['line'] = month_df['route_id'].map(ROUTE_LINE_MAP).fillna(month_df['route_id'])
        month_df.to_parquet(ckpt, index=False)
        print(f'  Saved {year}-{month:02d}: {len(month_df):,} rows  ({ckpt.stat().st_size/1e6:.1f} MB)')
        processed += 1
        del frames, month_df

print(f'\nDone. Processed={processed}  Skipped={skipped}  Failed days={failed_days}')


In [ ]:
# Verify all months saved
saved = sorted(STRATEGIC_DIR.glob('*.parquet'))
total_size = sum(f.stat().st_size for f in saved) / 1e6

print(f'Saved files : {len(saved)} months')
print(f'Total size  : {total_size:.1f} MB on disk')
print()

# Quick check: row counts per month
for f in saved:
    tmp = pd.read_parquet(f, columns=['route_id'])
    print(f'  {f.stem}: {len(tmp):,} rows')
    del tmp

print()
print('To load specific months in downstream notebooks:')
print('  pd.read_parquet(PROC_DIR / \'strategic\' / \'2024-07.parquet\')')
print('Or load multiple months:')
print('  pd.concat([pd.read_parquet(f) for f in sorted((PROC_DIR/\'strategic\').glob(\'2024-*.parquet\'))])')


## Summary

| Item | Result |
|------|--------|
| Available date range | 2019-09-15 → present |
| Analysis window | 2022-01-01 → 2026-05-31 |
| Lines covered | Red, Orange, Blue, Green B/C/D/E, Mattapan |
| Key columns confirmed | travel_time, dwell_time, headway_branch, headway_trunk, scheduled times |
| Jan 2024 sample | `data/processed/sample_jan2024.parquet` (885K rows) |
| Full aggregate | `data/processed/aggregate_full.parquet` (~4–5M rows, station×hour×route×date) |
| Strategic row-level sample | `data/processed/strategic/*.parquet` (每月一檔，2024-01 → 2026-05，共 29 個月) |

**Next:** `02_eda.ipynb` — EDA on aggregate_full for cross-year trends, then on strategic sample for regression.
